<a href="https://colab.research.google.com/github/chen323-ux/CTP/blob/main/PCR_data_processor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# Plate rows
plate_rows = list("ABCDEFGH")

# Prompt for group names vertically, one per plate row
# Example: Th0,Th17,Th0BL,Th17BL,Th17BL,,,
group_input = input(
    "Enter group names for rows A-H, separated by commas. "
    "Use blanks for empty rows:\n"
)

groups = [g.strip() for g in group_input.split(",")]

# Make sure there are exactly 8 rows
if len(groups) < 8:
    groups += [""] * (8 - len(groups))
elif len(groups) > 8:
    groups = groups[:8]

# Prompt for primer names across triplicate column groups
# Example: 18s,GPR65,GPR171,y
primer_input = input(
    "Enter primer names for column groups 1-3, 4-6, 7-9, 10-12:\n"
)

primers = [p.strip() for p in primer_input.split(",")]

if len(primers) != 4:
    raise ValueError("You must enter exactly 4 primer names.")

# Each primer occupies 3 columns
primer_column_blocks = {
    primers[0]: range(1, 4),    # columns 1-3
    primers[1]: range(4, 7),    # columns 4-6
    primers[2]: range(7, 10),   # columns 7-9
    primers[3]: range(10, 13)   # columns 10-12
}

records = []

for row_letter, group in zip(plate_rows, groups):
    if group == "":
        continue  # skip empty rows

    for primer, columns in primer_column_blocks.items():
        for replicate_number, column_number in enumerate(columns, start=1):
            well = f"{row_letter}{column_number}"

            records.append({
                "Well": well,
                "Group": group,
                "Primer": primer,
            })

final_df = pd.DataFrame(records)

final_df.head()


#test case: Cerebellum, Midbrain, Hippocampus, Cortex, PFC
#primers:  ,Ap1, ,GPR171

In [ ]:

# =========================
# Upload/read qPCR result file
# =========================

from google.colab import files

uploaded = files.upload()
qpcr_file = list(uploaded.keys())[0]

# Read Excel or CSV
if qpcr_file.endswith(".xlsx") or qpcr_file.endswith(".xls"):
    qpcr_df = pd.read_excel(qpcr_file)
elif qpcr_file.endswith(".csv"):
    qpcr_df = pd.read_csv(qpcr_file)
else:
    raise ValueError("Please upload an Excel or CSV file.")

# Clean column names
qpcr_df.columns = qpcr_df.columns.astype(str).str.strip()

# Check that needed columns exist
required_cols = ["Position", "Cq"]

for col in required_cols:
    if col not in qpcr_df.columns:
        raise ValueError(f"Could not find column: {col}")

# Keep only Position and Cq
cq_df = qpcr_df[["Position", "Cq"]].copy()

# Rename Position to Well so it matches final_df
cq_df = cq_df.rename(columns={"Position": "Well"})

# Clean well names
cq_df["Well"] = cq_df["Well"].astype(str).str.strip().str.upper()
final_df["Well"] = final_df["Well"].astype(str).str.strip().str.upper()

# Convert Cq values to numbers
cq_df["Cq"] = pd.to_numeric(cq_df["Cq"], errors="coerce")

# Remove blank wells if any
cq_df = cq_df[cq_df["Well"] != ""]

# Optional: check for duplicate well positions
duplicates = cq_df[cq_df["Well"].duplicated()]

if not duplicates.empty:
    print("Warning: duplicate well positions found:")
    display(duplicates)

# Merge Cq values into your current final_df
final_df_with_cq = final_df.merge(cq_df, on="Well", how="left")

# Show final result
final_df_with_cq.head()

In [ ]:
# ============================================================
# Detect and remove Cq outliers from technical triplicates
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Outlier thresholds
# ------------------------------------------------------------

MAX_TRIPLICATE_SD = 0.30

# The two agreeing technical replicates must be this close
MAX_CLOSE_PAIR_DIFFERENCE = 0.30

# The suspected outlier must differ from the mean of the
# closest pair by more than this amount
MIN_OUTLIER_DIFFERENCE = 0.50


# ------------------------------------------------------------
# Make a copy so the original dataframe is not changed
# ------------------------------------------------------------

cq_qc_df = final_df_with_cq.copy()

# Make sure Cq is numeric
cq_qc_df["Cq"] = pd.to_numeric(
    cq_qc_df["Cq"],
    errors="coerce"
)

# Add QC columns
cq_qc_df["Cq_Outlier"] = False
cq_qc_df["QC_Status"] = "Kept"


# Store information for the QC report
qc_results = []


# ------------------------------------------------------------
# Analyze each Group × Primer technical triplicate
# ------------------------------------------------------------

for (group, primer), group_indices in cq_qc_df.groupby(
    ["Group", "Primer"],
    dropna=False
).groups.items():

    # Get the Cq values for the current Group × Primer
    cq_values = (
        cq_qc_df
        .loc[group_indices, "Cq"]
        .dropna()
    )

    number_of_valid_cqs = len(cq_values)


    # --------------------------------------------------------
    # Only evaluate complete technical triplicates
    # --------------------------------------------------------

    if number_of_valid_cqs != 3:

        qc_results.append({
            "Group": group,
            "Primer": primer,
            "Number of valid Cq values": number_of_valid_cqs,
            "Cq values": ", ".join(
                f"{value:.2f}"
                for value in cq_values
            ),
            "Triplicate SD": np.nan,
            "Closest-pair difference": np.nan,
            "Third-value difference": np.nan,
            "Removed well": None,
            "Removed Cq": np.nan,
            "QC decision":
                f"Not evaluated: expected 3 Cq values, found "
                f"{number_of_valid_cqs}"
        })

        continue


    # Convert the three values into a NumPy array
    values = cq_values.to_numpy(dtype=float)


    # --------------------------------------------------------
    # Calculate triplicate SD
    # --------------------------------------------------------

    triplicate_sd = np.std(
        values,
        ddof=1
    )


    # --------------------------------------------------------
    # Compare all possible pairs
    # --------------------------------------------------------

    possible_pairs = [
        (0, 1),
        (0, 2),
        (1, 2)
    ]

    # Find the pair with the smallest Cq difference
    closest_pair = min(
        possible_pairs,
        key=lambda pair:
            abs(values[pair[0]] - values[pair[1]])
    )

    first_position = closest_pair[0]
    second_position = closest_pair[1]

    closest_pair_difference = abs(
        values[first_position]
        -
        values[second_position]
    )


    # --------------------------------------------------------
    # Find the remaining value
    # --------------------------------------------------------

    all_positions = {0, 1, 2}

    outlier_position = list(
        all_positions
        -
        set(closest_pair)
    )[0]


    # Mean Cq of the two closest technical replicates
    closest_pair_mean = np.mean([
        values[first_position],
        values[second_position]
    ])


    # Distance between the third value and the closest-pair mean
    third_value_difference = abs(
        values[outlier_position]
        -
        closest_pair_mean
    )


    # --------------------------------------------------------
    # Determine whether the third value should be removed
    # --------------------------------------------------------

    remove_outlier = (

        triplicate_sd > MAX_TRIPLICATE_SD

        and

        closest_pair_difference
        <= MAX_CLOSE_PAIR_DIFFERENCE

        and

        third_value_difference
        > MIN_OUTLIER_DIFFERENCE
    )


    # --------------------------------------------------------
    # Flag the outlier
    # --------------------------------------------------------

    if remove_outlier:

        outlier_dataframe_index = (
            cq_values.index[outlier_position]
        )

        removed_well = cq_qc_df.loc[
            outlier_dataframe_index,
            "Well"
        ]

        removed_cq = cq_qc_df.loc[
            outlier_dataframe_index,
            "Cq"
        ]

        cq_qc_df.loc[
            outlier_dataframe_index,
            "Cq_Outlier"
        ] = True

        cq_qc_df.loc[
            outlier_dataframe_index,
            "QC_Status"
        ] = "Removed as possible technical outlier"

        qc_decision = "One possible outlier removed"


    else:

        removed_well = None
        removed_cq = np.nan

        qc_decision = "All three Cq values kept"


    # --------------------------------------------------------
    # Add results to QC report
    # --------------------------------------------------------

    qc_results.append({

        "Group": group,

        "Primer": primer,

        "Number of valid Cq values":
            number_of_valid_cqs,

        "Cq values":
            ", ".join(
                f"{value:.2f}"
                for value in values
            ),

        "Triplicate SD":
            triplicate_sd,

        "Closest-pair difference":
            closest_pair_difference,

        "Third-value difference":
            third_value_difference,

        "Removed well":
            removed_well,

        "Removed Cq":
            removed_cq,

        "QC decision":
            qc_decision
    })


# ============================================================
# Create final cleaned dataframe
# ============================================================

final_df_with_cq_clean = (

    cq_qc_df[
        cq_qc_df["Cq_Outlier"] == False
    ]

    .copy()

    .reset_index(drop=True)
)


# ============================================================
# Create QC report
# ============================================================

cq_qc_report = pd.DataFrame(
    qc_results
)


# Round numerical QC values
columns_to_round = [

    "Triplicate SD",

    "Closest-pair difference",

    "Third-value difference",

    "Removed Cq"
]

cq_qc_report[
    columns_to_round
] = (

    cq_qc_report[
        columns_to_round
    ]

    .round(3)
)


# ============================================================
# Display results
# ============================================================

number_removed = int(
    cq_qc_df["Cq_Outlier"].sum()
)

print(
    f"Number of possible Cq outliers removed: "
    f"{number_removed}"
)


print("\nQC report:")

display(
    cq_qc_report
)


print("\nRemoved wells:")

display(

    cq_qc_df[

        cq_qc_df[
            "Cq_Outlier"
        ]

    ][

        [
            "Well",
            "Group",
            "Primer",
            "Cq",
            "QC_Status"
        ]

    ]
)


print("\nCleaned qPCR dataframe:")

display(
    final_df_with_cq_clean.head()
)

In [ ]:
import pandas as pd

# Make sure Cq is numeric
final_df_with_cq["Cq"] = pd.to_numeric(final_df_with_cq["Cq"], errors="coerce")

# Extract plate row from Well if Plate Row column does not already exist
# Example: A1 -> A, D5 -> D
if "Plate Row" not in final_df_with_cq.columns:
    final_df_with_cq["Plate Row"] = (
        final_df_with_cq["Well"]
        .astype(str)
        .str.extract(r"([A-Ha-h])")[0]
        .str.upper()
    )

# Clean primer names
final_df_with_cq["Primer"] = final_df_with_cq["Primer"].astype(str).str.strip()

print("Available primers:")
print(final_df_with_cq["Primer"].dropna().unique())

# Ask only once for the whole PCR plate
housekeeping_primer = input("What is the housekeeping gene primer for this PCR plate? ").strip()

# Find the housekeeping rows
hk_rows = final_df_with_cq[
    final_df_with_cq["Primer"].str.lower() == housekeeping_primer.lower()
].copy()

# Calculate housekeeping average separately for each plate row
# This prevents D Th17BL and E Th17BL from being averaged together
hk_avg_df = (
    hk_rows
    .groupby(["Plate Row"], as_index=False)["Cq"]
    .mean()
    .rename(columns={"Cq": "Housekeeping Average"})
)

# Remove old columns if you rerun this cell
final_df_with_cq = final_df_with_cq.drop(
    columns=["Housekeeping Primer", "Housekeeping Average"],
    errors="ignore"
)

# Add housekeeping primer name
final_df_with_cq["Housekeeping Primer"] = housekeeping_primer

# Merge row-specific housekeeping average back into main dataframe
final_df_with_cq = final_df_with_cq.merge(
    hk_avg_df,
    on="Plate Row",
    how="left"
)

final_df_with_cq.head()

In [ ]:
# Make sure both columns are numeric
final_df_with_cq["Cq"] = pd.to_numeric(final_df_with_cq["Cq"], errors="coerce")
final_df_with_cq["Housekeeping Average"] = pd.to_numeric(
    final_df_with_cq["Housekeeping Average"],
    errors="coerce"
)

# Create Target Control column
final_df_with_cq["Target Control"] = (
    final_df_with_cq["Cq"] - final_df_with_cq["Housekeeping Average"]
)

# Create Fold Change column
final_df_with_cq["Fold Change"] = (
    2 ** (-(final_df_with_cq["Target Control"]))
)

# Create Average Amplication column
# Extract Plate Row from Well if it does not already exist
# Example: A1 -> A, D5 -> D
if "Plate Row" not in final_df_with_cq.columns:
    final_df_with_cq["Plate Row"] = (
        final_df_with_cq["Well"]
        .astype(str)
        .str.extract(r"([A-Ha-h])")[0]
        .str.upper()
    )

# Make sure Fold Change is numeric
final_df_with_cq["Fold Change"] = pd.to_numeric(
    final_df_with_cq["Fold Change"],
    errors="coerce"
)

# Create Average Amplification column
# This averages each triplicate set within the same plate row and primer
final_df_with_cq["Average Amplification"] = (
    final_df_with_cq
    .groupby(["Plate Row", "Primer"])["Fold Change"]
    .transform("mean")
)

final_df_with_cq.head()


In [ ]:
# Make Average Amplification numeric
final_df_with_cq["Average Amplification"] = pd.to_numeric(
    final_df_with_cq["Average Amplification"],
    errors="coerce"
)

# Clean Group names
final_df_with_cq["Group"] = (
    final_df_with_cq["Group"]
    .astype("string")
    .str.strip()
)

# Clean Primer names
final_df_with_cq["Primer"] = (
    final_df_with_cq["Primer"]
    .astype("string")
    .str.strip()
)

# Remove:
# 1. Missing primer names
# 2. Blank primer names
# 3. The housekeeping primer
target_df = final_df_with_cq[
    final_df_with_cq["Primer"].notna()
    &
    final_df_with_cq["Primer"].ne("")
    &
    final_df_with_cq["Primer"].str.lower().ne(
        housekeeping_primer.lower()
    )
].copy()

In [ ]:
# Collapse repeated triplicate Average Amplification values
row_level_summary = (
    target_df
    .groupby(
        ["Plate Row", "Group", "Primer"],
        as_index=False
    )["Average Amplification"]
    .mean()
)

# Create final wide dataframe
final_summary_df = (
    row_level_summary
    .groupby(
        ["Group", "Primer"],
        as_index=False
    )["Average Amplification"]
    .mean()
    .pivot(
        index="Group",
        columns="Primer",
        values="Average Amplification"
    )
)

# Preserve original group order
group_order = (
    target_df["Group"]
    .dropna()
    .drop_duplicates()
    .tolist()
)

final_summary_df = (
    final_summary_df
    .reindex(group_order)
    .dropna(axis=0, how="all")
    .dropna(axis=1, how="all")
)

final_summary_df

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Create one graph for each actual primer
for primer in final_summary_df.columns:

    # Skip blank primer names
    if pd.isna(primer) or str(primer).strip() == "":
        continue

    # Keep only nonmissing values for this primer
    graph_df = final_summary_df[[primer]].dropna()

    # Skip columns with no values
    if graph_df.empty:
        continue

    # Create graph
    plt.figure(figsize=(6, 4))

    plt.bar(
        graph_df.index,
        graph_df[primer]
    )

    plt.title(f"{primer} Average Amplification")
    plt.xlabel("Group")
    plt.ylabel("Average Amplification")

    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()

    # Create a filename that does not contain invalid characters
    safe_primer_name = (
        str(primer)
        .replace("/", "_")
        .replace("\\", "_")
        .replace(" ", "_")
    )

    graph_file = (
        f"{safe_primer_name}_"
        "average_amplification_bar_graph.png"
    )

    # Save graph
    plt.savefig(
        graph_file,
        dpi=300,
        bbox_inches="tight"
    )

    # Display graph
    plt.show()

    # Close graph to prevent figures from accumulating
    plt.close()

    print(f"Saved graph: {graph_file}")